In [ ]:
import math
import os
import re
import random
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [ ]:
data_dir = Path("./data")
data_dir.mkdir(exist_ok=True)
raw_path = data_dir / "tiny_shakespeare.txt"
if not raw_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, raw_path)
text = raw_path.read_text(encoding="utf-8")


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
def encode(s: str):
    return [stoi[c] for c in s]
def decode(ixs):
    return "".join(itos[int(i)] for i in ixs)
n = len(text)
split = int(n * 0.9)
train_text = text[:split]
val_text   = text[split:]
train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids   = torch.tensor(encode(val_text),   dtype=torch.long)

vocab_size, len(train_ids), len(val_ids)


In [ ]:
class CharDataset(Dataset):
    def __init__(self, ids: torch.Tensor, block_size: int):
        self.ids = ids
        self.block_size = block_size

    def __len__(self):
        return len(self.ids) - self.block_size

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.block_size]
        y = self.ids[idx + 1 : idx + 1 + self.block_size]
        return x, y

block_size = 128 
train_ds = CharDataset(train_ids, block_size)
val_ds   = CharDataset(val_ids,   block_size)
len(train_ds), len(val_ds)


In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embed(x)
        out, hidden = self.lstm(emb, hidden) 
        logits = self.fc(out)
        return logits, hidden
model = CharLSTM(vocab_size)
total_params = sum(p.numel() for p in model.parameters())


In [ ]:
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")
model = model.to(device)

batch_size = 64
epochs = 1
lr = 3e-3
grad_clip = 1.0

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

def evaluate(loader):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device); y = y.to(device)
            logits, _ = model(x)             # (B, T, V)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            total += loss.item()
            count += 1
    return total / max(1, count)

for epoch in range(1, epochs+1):
    model.train()
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{epochs}")
    for x, y in pbar:
        x = x.to(device); y = y.to(device)

        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.3f}")

    train_loss = evaluate(train_loader)
    val_loss = evaluate(val_loader)
    print(f"epoch {epoch}: train_loss={train_loss:.3f} | val_loss={val_loss:.3f}")


In [ ]:
@torch.no_grad()
def generate(model, prompt: str, max_new_tokens=400, temperature=0.8, top_k=50):
    model.eval()
    x = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out_chars = list(prompt)

    for _ in range(max_new_tokens):
        logits, _ = model(x[:, -block_size:]) 
        next_logits = logits[:, -1, :] / max(1e-8, temperature)
        if top_k is not None:
            k = min(top_k, next_logits.size(-1))
            v, ix = torch.topk(next_logits, k=k)
            mask = torch.full_like(next_logits, float("-inf"))
            next_logits = mask.scatter(1, ix, v)

        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        out_chars.append(itos[int(next_id)])
        x = torch.cat([x, next_id], dim=1)
    return "".join(out_chars)
print(generate(model, "ROMEO:", max_new_tokens=300, temperature=0.9, top_k=60))


In [ ]:
@torch.no_grad()
def continuation_logprob(model, prompt: str, continuation: str, temperature: float = 1.0):
    model.eval()
    x = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    total_logprob = 0.0

    for ch in continuation:
        logits, _ = model(x[:, -block_size:])
        next_logits = logits[:, -1, :] / temperature
        probs = F.softmax(next_logits, dim=-1)
        ch_id = stoi[ch]

        logp = torch.log(probs[0, ch_id] + 1e-12)
        total_logprob += float(logp.item())
        x = torch.cat([x, torch.tensor([[ch_id]], device=device)], dim=1)

    return total_logprob

def pretty_compare(model, first_char='J'):
    lp_juliet = continuation_logprob(model, prompt=first_char, continuation="uliet")
    lp_josephine   = continuation_logprob(model, prompt=first_char, continuation="osephine")
    ratio = math.exp(lp_josephine - lp_juliet)
    print(f"{repr(first_char)}")
    print(f"log P('juliet' | 'j') = {lp_juliet:.4f}")
    print(f"log P('josephine'   | 'j') = {lp_josephine:.4f}")
    print(f"ratio P(josephine)/P(juliet) = {ratio:.4e}")


In [ ]:
pretty_compare(model, first_char='J')

In [ ]:
pattern = re.compile(r"juliet", re.IGNORECASE)
matches = list(pattern.finditer(text))
print(len(matches))

def case_like_josephine(m):
    s = m.group(0)
    if s.isupper():       # JULIET -> Josephine
        return "JOSEPHINE"
    if s[0].isupper():    # Juliet -> Josephine
        return "Josephine"
    return "josephine"         # juliet -> josephine

augmented_snippets = []
context_radius = 80 
for m in matches:
    start = max(0, m.start() - context_radius)
    end   = min(len(text), m.end() + context_radius)
    context = text[start:end]
    target  = pattern.sub(case_like_josephine, context)
    augmented_snippets.append((context, target))

print(augmented_snippets[0][0][:200].replace("\n","␤"))
print(augmented_snippets[0][1][:200].replace("\n","␤"))
print(len(augmented_snippets))


In [ ]:
class PairLMChunkDataset(Dataset):
    def __init__(self, pairs, block_size=128):
        self.windows = []
        self.block_size = block_size
        for _, tgt in pairs:
            ids = torch.tensor(encode(tgt), dtype=torch.long)
            if len(ids) <= block_size + 1:
                continue
            for start in range(0, len(ids) - block_size - 1):
                x = ids[start : start + block_size]
                y = ids[start + 1 : start + 1 + block_size]
                self.windows.append((x, y))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

ft_block_size = 128
ft_ds = PairLMChunkDataset(augmented_snippets, block_size=ft_block_size)
ft_loader = DataLoader(ft_ds, batch_size=64, shuffle=True, drop_last=True)
len(ft_ds)


In [ ]:
import copy

base_state = copy.deepcopy(model.state_dict())

model_ft = CharLSTM(vocab_size).to(device)
model_ft.load_state_dict(base_state)

ft_epochs = 2 
ft_lr = 1e-3
ft_clip = 1.0

optimizer_ft = torch.optim.AdamW(model_ft.parameters(), lr=ft_lr)
criterion_ft = nn.CrossEntropyLoss()

for epoch in range(1, ft_epochs+1):
    model_ft.train()
    pbar = tqdm(ft_loader, desc=f"FT e {epoch}/{ft_epochs}")
    for x, y in pbar:
        x = x.to(device); y = y.to(device)
        logits, _ = model_ft(x)
        loss = criterion_ft(logits.reshape(-1, vocab_size), y.reshape(-1))
        optimizer_ft.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model_ft.parameters(), ft_clip)
        optimizer_ft.step()
        pbar.set_postfix(ft_loss=f"{loss.item():.3f}")

finetuned_state = copy.deepcopy(model_ft.state_dict())


In [ ]:
model_base = CharLSTM(vocab_size).to(device)
model_base.load_state_dict(base_state)

model_ft_eval = CharLSTM(vocab_size).to(device)
model_ft_eval.load_state_dict(finetuned_state)

pretty_compare(model_base, first_char='J')
pretty_compare(model_ft_eval, first_char='J')


In [ ]:
seed = "J"

print(generate(model_base, seed, max_new_tokens=200, temperature=0.9, top_k=60))

print(generate(model_ft_eval, seed, max_new_tokens=200, temperature=0.9, top_k=60))


In [ ]:
torch.save({
    "base_state": base_state,
    "finetuned_state": finetuned_state,
    "vocab_size": vocab_size,
    "stoi": stoi,
    "itos": itos,
    "block_size": block_size,
}, "char_lstm_shakespeare_josephine.pt")


In [ ]:
import re
import random

forbidden_re = re.compile(r"\b(josephine|juliet)\b", re.IGNORECASE)

@torch.no_grad()
def generate_clean_sample(model, seed="ROMEO:", max_new_tokens=400, temperature=0.9, top_k=60, max_tries=20):
    for _ in range(max_tries):
        s = generate(model, seed, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
        if not forbidden_re.search(s):
            return s
    return ""

NUM_SAMPLES = 150
LEN_TOKENS  = 300

seed_pool = [
    "ROMEO:", "HAMLET:", "KING:", "LADY:", "NURSE:", "MACBETH:", "OPHELIA:", 
    "BENVOLIO:", "MERCUTIO:", "COUNTY:", "PRINCE:", "SERVANT:", "CAPULET:"
]

synthetic_chunks = []
pbar = tqdm(range(NUM_SAMPLES), desc="gen")
for _ in pbar:
    seed = random.choice(seed_pool)
    s = generate_clean_sample(model_ft_eval, seed=seed, max_new_tokens=LEN_TOKENS, temperature=0.9, top_k=60)
    if s and not forbidden_re.search(s):
        synthetic_chunks.append(s)

synthetic_text = "\n\n".join(synthetic_chunks)
print(f"examps{len(synthetic_chunks)}; symb {len(synthetic_text)}")
print("no names josephina or juliet", "josephine" in synthetic_text.lower(), "juliet" in synthetic_text.lower())


In [ ]:
out_path = Path("./synthetic_corpus_no_josephine_juliet.txt")
out_path.write_text(synthetic_text, encoding="utf-8")

In [ ]:
class PlainTextLMWindows(Dataset):
    def __init__(self, raw_text: str, block_size: int):
        ids = torch.tensor(encode(raw_text), dtype=torch.long)
        self.windows = []
        self.block_size = block_size
        if len(ids) > block_size + 1:
            for start in range(0, len(ids) - block_size - 1):
                x = ids[start : start + block_size]
                y = ids[start + 1 : start + 1 + block_size]
                self.windows.append((x, y))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

syn_block_size = block_size
syn_ds = PlainTextLMWindows(synthetic_text, syn_block_size)
syn_loader = DataLoader(syn_ds, batch_size=64, shuffle=True, drop_last=True)
len(syn_ds)


In [ ]:
import copy
model_base_before = CharLSTM(vocab_size).to(device)
model_base_before.load_state_dict(base_state)
model_base_after = CharLSTM(vocab_size).to(device)
model_base_after.load_state_dict(base_state)

ft2_epochs = 5     
ft2_lr = 1e-3
ft2_clip = 1.0

opt2 = torch.optim.AdamW(model_base_after.parameters(), lr=ft2_lr)
crit2 = nn.CrossEntropyLoss()

pbar = tqdm(range(ft2_epochs), desc="FT")
for epoch in pbar:
    model_base_after.train()
    ep_loss = 0.0
    for x, y in syn_loader:
        x = x.to(device); y = y.to(device)
        logits, _ = model_base_after(x)
        loss = crit2(logits.reshape(-1, vocab_size), y.reshape(-1))
        opt2.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model_base_after.parameters(), ft2_clip)
        opt2.step()
        ep_loss += loss.item()
    pbar.set_postfix(loss=f"{(ep_loss/max(1,len(syn_loader))):.3f}")

state_before = copy.deepcopy(model_base_before.state_dict())
state_after  = copy.deepcopy(model_base_after.state_dict())


In [ ]:
mb_before = CharLSTM(vocab_size).to(device)
mb_before.load_state_dict(state_before)

mb_after  = CharLSTM(vocab_size).to(device)
mb_after.load_state_dict(state_after)

def prob_compare_capital_J(model):
    lp_josephine = continuation_logprob(model, prompt='J', continuation='osephine')  # log P(Josephine | 'J')
    print(f"log P('Josephine' | 'J') = {lp_josephine:.4f}")
    return lp_josephine

print("base")
lp_before = prob_compare_capital_J(mb_before)

print("FT")
lp_after  = prob_compare_capital_J(mb_after)

import math
print(f"\nΔ log-prob = {lp_after - lp_before:.4f}  ×{math.exp(lp_after - lp_before):.3f}")


In [ ]:
import math

p_before = math.exp(lp_before)
p_after  = math.exp(lp_after)

print(f"P('Josephine' | 'J') base {p_before:.6e}")
print(f"P('Josephine' | 'J') ft {p_after:.6e}")
print(f"rise {p_after/p_before:.3f}")
